In [ ]:
import sys
sys.path.insert(0, "/content/qp_ai_vision/src")
import torch

from rsw_ai.model.SsdDataset import SsdDataset
from rsw_ai.model.Dataset import Dataset
from rsw_ai.backend.import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset import import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset

from rsw_ai.backend.VisionToSsdConverter import VisionToSsdConverter
from rsw_ai.model.SsdTorchDataset import SsdTorchDataset
import torchvision
from rsw_ai.model.SsdTrainer import SsdTrainer
from rsw_ai.backend.ssdTransform import SsdTransform
from rsw_ai.mapping.VisionYoloDataset_to_VisionDetectionDataset import (
    VisionYoloDataset_to_VisionDetectionDataset
)
from rsw_ai.enum.DatasetRepositoryLayout import (
   DatasetRepositoryLayout
)
KAGGLE_DOWNLOADED_PATH = "outputs/datasets/downloads/kaggle/"  # The main directory after downloading from the Kaggle dataset.

dataset_repository_layout = (
    DatasetRepositoryLayout.NADINPETHIYAGODA_VEHICLE_DATASET_FOR_YOLO
)

DIR_DOWNLOADED = (
    "../"  # backward one dir
    + KAGGLE_DOWNLOADED_PATH
    + dataset_repository_layout.label
    + "/"
    + "/vehicle dataset"
)
print(DIR_DOWNLOADED)

def main():
  yolo_data = (
    import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset(
        DIR_DOWNLOADED
    )
  )


# ============================================================
# 2. YOLO Dataset -> Detection Dataset
# ============================================================

  detection_data = (
    VisionYoloDataset_to_VisionDetectionDataset(
        yolo_data
    )
  )


# ============================================================
# 3. Detection Dataset -> SSD Dataset
# ============================================================

  converter = VisionToSsdConverter()

  ssd_data = converter.convert(
    detection_data
  )

  transfrom = SsdTransform(
    resize=(300, 300),
    horizontal_flip=False,
    vertical_flip=False,
    brightness=1.0,
    contrast=1.0,
    normalize=True
  )

  train_dataset = SsdTorchDataset(ssd_data,transfrom,split="train")

##edges，textures，shapes，higher-level features，shape
  model = torchvision.models.detection.ssd300_vgg16(
    num_classes=len(ssd_data.class_map) + 1,
    score_thresh=0.5,
    nms_thresh=0.3,
    detections_per_img=100
  )

  optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005,
  )

  trainer = SsdTrainer()


  trainer.train(
        train_dataset=train_dataset,
        model=model,
        optimizer=optimizer,
        epochs=50,
  )

if __name__ == "__main__":
    main()



../outputs/datasets/downloads/kaggle/nadinpethiyagoda_vehicle_dataset_for_yolo//vehicle dataset
Downloading: "https://download.pytorch.org/models/vgg16_features-amdegroot-88682ab5.pth" to C:\Users\vsgm0/.cache\torch\hub\checkpoints\vgg16_features-amdegroot-88682ab5.pth


100%|██████████| 528M/528M [02:39<00:00, 3.48MB/s] 


Training on: cpu
Epoch 1/50
Batch 0, Loss: 490.1043
